In [9]:
import os

print(os.listdir("/kaggle/input/competitions"))

['plant-pathology-2021-fgvc8']


In [14]:
DATA_DIR = "/kaggle/input/competitions/plant-pathology-2021-fgvc8"

In [15]:
import os



print("Train images:", len(os.listdir(f"{DATA_DIR}/train_images")))
print("Test images:", len(os.listdir(f"{DATA_DIR}/test_images")))

Train images: 18632
Test images: 3


In [16]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score

SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

KAGGLE_DIR = "/kaggle/input/plant-pathology-2021-fgvc8"
LOCAL_DIR = "/content/plant_pathology"
DATA_DIR = KAGGLE_DIR if os.path.exists(KAGGLE_DIR) else LOCAL_DIR

TRAIN_CSV = os.path.join(DATA_DIR, "train.csv")
TRAIN_IMG_DIR = os.path.join(DATA_DIR, "train_images")
TEST_IMG_DIR = os.path.join(DATA_DIR, "test_images")
SAMPLE_SUB = os.path.join(DATA_DIR, "sample_submission.csv")

IMG_SIZE = 380
BATCH_SIZE = 16
EPOCHS_HEAD = 8
EPOCHS_FINETUNE = 14

In [19]:
import os

for root, dirs, files in os.walk("/kaggle/input/competitions"):
    if "train.csv" in files:
        print(os.path.join(root, "train.csv"))

/kaggle/input/competitions/plant-pathology-2021-fgvc8/train.csv


In [ ]:
TRAIN_CSV = "/kaggle/input/competitions/plant-pathology-2021-fgvc8/train.csv"

In [28]:
df = pd.read_csv(TRAIN_CSV)

df["label_list"] = df["labels"].apply(lambda x: x.split(" "))

mlb = MultiLabelBinarizer()
y = mlb.fit_transform(df["label_list"])

CLASSES = mlb.classes_
NUM_CLASSES = len(CLASSES)

strat_key = df["labels"]

df_labels = pd.DataFrame(y, columns=CLASSES)

df = pd.concat([df[["image"]], df_labels], axis=1)

print(df.shape, CLASSES)

(18632, 7) ['complex' 'frog_eye_leaf_spot' 'healthy' 'powdery_mildew' 'rust' 'scab']


In [48]:
TRAIN_IMG_DIR = "/kaggle/input/competitions/plant-pathology-2021-fgvc8/train_images"

raw = pd.read_csv(TRAIN_CSV)
strat_col = raw["labels"]

try:
    train_df, val_df = train_test_split(
        df, test_size=0.2, random_state=SEED, stratify=strat_col
    )
except ValueError:
    train_df, val_df = train_test_split(
        df, test_size=0.2, random_state=SEED
    )

train_paths = (TRAIN_IMG_DIR + "/" + train_df["image"]).values
val_paths = (TRAIN_IMG_DIR + "/" + val_df["image"]).values

train_labels = train_df[CLASSES].values.astype("float32")
val_labels = val_df[CLASSES].values.astype("float32")

print(len(train_paths), len(val_paths))
print(val_paths[0])

14905 3727
/kaggle/input/competitions/plant-pathology-2021-fgvc8/train_images/92e83ad88015df3f.jpg


In [50]:
AUTOTUNE = tf.data.AUTOTUNE

def load_image(path, label=None, training=False):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    if training:
        img = tf.image.random_flip_left_right(img)
        img = tf.image.random_flip_up_down(img)
        img = tf.image.rot90(img, k=tf.random.uniform([], 0, 4, dtype=tf.int32))
        img = tf.image.random_brightness(img, 0.1)
        img = tf.image.random_contrast(img, 0.9, 1.1)
    img = tf.keras.applications.efficientnet.preprocess_input(img)
    if label is None:
        return img
    return img, label

def make_dataset(paths, labels=None, training=False):
    if labels is not None:
        ds = tf.data.Dataset.from_tensor_slices((paths, labels))
        ds = ds.map(lambda p, l: load_image(p, l, training), num_parallel_calls=AUTOTUNE)
    else:
        ds = tf.data.Dataset.from_tensor_slices(paths)
        ds = ds.map(lambda p: load_image(p, None, training), num_parallel_calls=AUTOTUNE)
    if training:
        ds = ds.shuffle(1024, seed=SEED)
    ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return ds

train_ds = make_dataset(train_paths, train_labels, training=True)
val_ds = make_dataset(val_paths, val_labels, training=False)

In [18]:
base_model = tf.keras.applications.EfficientNetB7(
    include_top=False,
    weights="imagenet",
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)
base_model.trainable = False

inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = base_model(inputs, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.3)(x)
outputs = tf.keras.layers.Dense(NUM_CLASSES, activation="sigmoid")(x)
model = tf.keras.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=[
        tf.keras.metrics.BinaryAccuracy(name="accuracy"),
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall"),
    ]
)
model.summary()

258076736/258076736 ━━━━━━━━━━━━━━━━━━━━ 7s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 380, 380, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb7 (Functional)     │ (None, 12, 12, 2560)   │    64,097,687 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 2560)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 2560)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 6)              │        15,366 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 64,113,053 (244.57 MB)

 Trainable params: 15,366 (60.02 KB)

 Non-trainable params: 64,097,687 (244.51 MB)

In [19]:
checkpoint_head = tf.keras.callbacks.ModelCheckpoint(
    "best_head.keras", monitor="val_loss", save_best_only=True, mode="min"
)
early_stop_head = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=3, restore_best_weights=True
)

history_head = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_HEAD,
    callbacks=[checkpoint_head, early_stop_head]
)

Epoch 1/8


2026-09-14 15:38:55.894039: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-14 15:38:56.171703: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-14 15:38:57.033322: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-14 15:38:57.221288: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-14 15:38:58.840052: E external/local_xla/xla/stream_

931/932 ━━━━━━━━━━━━━━━━━━━━ 0s 332ms/step - accuracy: 0.8527 - loss: 0.3406 - precision: 0.6975 - recall: 0.3128

2026-09-14 15:44:57.605404: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-14 15:44:57.827298: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-14 15:44:58.536963: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-14 15:44:58.701685: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-14 15:45:00.307702: E external/local_xla/xla/stream_

932/932 ━━━━━━━━━━━━━━━━━━━━ 0s 384ms/step - accuracy: 0.8528 - loss: 0.3405 - precision: 0.6975 - recall: 0.3129

2026-09-14 15:47:15.412865: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-14 15:47:15.686466: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-14 15:47:16.646131: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-14 15:47:16.834120: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-14 15:47:18.551115: E external/local_xla/xla/stream_

932/932 ━━━━━━━━━━━━━━━━━━━━ 592s 521ms/step - accuracy: 0.8733 - loss: 0.2966 - precision: 0.7633 - recall: 0.4327 - val_accuracy: 0.8970 - val_loss: 0.2425 - val_precision: 0.8198 - val_recall: 0.5508
Epoch 2/8
932/932 ━━━━━━━━━━━━━━━━━━━━ 410s 422ms/step - accuracy: 0.8986 - loss: 0.2438 - precision: 0.7964 - recall: 0.5890 - val_accuracy: 0.9079 - val_loss: 0.2209 - val_precision: 0.8166 - val_recall: 0.6317
Epoch 3/8
932/932 ━━━━━━━━━━━━━━━━━━━━ 412s 424ms/step - accuracy: 0.9047 - loss: 0.2305 - precision: 0.8042 - recall: 0.6243 - val_accuracy: 0.9114 - val_loss: 0.2113 - val_precision: 0.8266 - val_recall: 0.6446
Epoch 4/8
932/932 ━━━━━━━━━━━━━━━━━━━━ 412s 424ms/step - accuracy: 0.9083 - loss: 0.2219 - precision: 0.8064 - recall: 0.6478 - val_accuracy: 0.9134 - val_loss: 0.2071 - val_precision: 0.8198 - val_recall: 0.6669
Epoch 5/8
932/932 ━━━━━━━━━━━━━━━━━━━━ 410s 422ms/step - accuracy: 0.9109 - loss: 0.2175 - precision: 0.8115 - recall: 0.6598 - val_accuracy: 0.9180 - val_los

In [20]:
checkpoint_head = tf.keras.callbacks.ModelCheckpoint(
    "best_head_finetuned.keras", monitor="val_loss", save_best_only=True, mode="min"
)
early_stop_head = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=3, restore_best_weights=True
)

base_model.trainable = True
for layer in base_model.layers[:-80]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="binary_crossentropy",
    metrics=[
        tf.keras.metrics.BinaryAccuracy(name="accuracy"),
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall"),
    ]
)


reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss", factor=0.5, patience=2, min_lr=1e-7
)

history_ft = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_FINETUNE,
    callbacks=[checkpoint_head, early_stop_head, reduce_lr]
)

Epoch 1/14


2026-09-14 16:44:14.932353: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-14 16:44:15.068445: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-14 16:44:15.398066: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-14 16:44:15.569164: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-14 16:44:16.194819: E external/local_xla/xla/stream_

931/932 ━━━━━━━━━━━━━━━━━━━━ 0s 422ms/step - accuracy: 0.8275 - loss: 0.3935 - precision: 0.5369 - recall: 0.7661

2026-09-14 16:51:41.016884: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-14 16:51:41.159250: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-14 16:51:41.403917: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-14 16:51:41.575472: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-14 16:51:41.854507: E external/local_xla/xla/stream_

932/932 ━━━━━━━━━━━━━━━━━━━━ 674s 598ms/step - accuracy: 0.8799 - loss: 0.2883 - precision: 0.6503 - recall: 0.7245 - val_accuracy: 0.9340 - val_loss: 0.1760 - val_precision: 0.8190 - val_recall: 0.8148 - learning_rate: 1.0000e-05
Epoch 2/14
932/932 ━━━━━━━━━━━━━━━━━━━━ 497s 515ms/step - accuracy: 0.9274 - loss: 0.1856 - precision: 0.8340 - recall: 0.7466 - val_accuracy: 0.9429 - val_loss: 0.1487 - val_precision: 0.8559 - val_recall: 0.8224 - learning_rate: 1.0000e-05
Epoch 3/14
932/932 ━━━━━━━━━━━━━━━━━━━━ 496s 514ms/step - accuracy: 0.9356 - loss: 0.1630 - precision: 0.8553 - recall: 0.7742 - val_accuracy: 0.9475 - val_loss: 0.1371 - val_precision: 0.8621 - val_recall: 0.8440 - learning_rate: 1.0000e-05
Epoch 4/14
932/932 ━━━━━━━━━━━━━━━━━━━━ 496s 514ms/step - accuracy: 0.9420 - loss: 0.1466 - precision: 0.8700 - recall: 0.7979 - val_accuracy: 0.9509 - val_loss: 0.1267 - val_precision: 0.8737 - val_recall: 0.8512 - learning_rate: 1.0000e-05
Epoch 5/14
932/932 ━━━━━━━━━━━━━━━━━━━━ 497

: 

In [51]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    if "best_head_finetuned.keras" in files:
        MODEL_PATH = os.path.join(root, "best_head_finetuned.keras")
        print(MODEL_PATH)
        break

model = tf.keras.models.load_model(MODEL_PATH)

/kaggle/input/datasets/mohamedadiab/model7/best_head_finetuned.keras


In [46]:
TRAIN_IMG_DIR = "/kaggle/input/competitions/plant-pathology-2021-fgvc8/train_images"

In [52]:
val_probs = model.predict(val_ds)

def optimize_thresholds(y_true, y_prob, num_classes):
    thresholds = np.full(num_classes, 0.5)
    grid = np.arange(0.1, 0.91, 0.02)
    for c in range(num_classes):
        best_t, best_f1 = 0.5, -1.0
        for t in grid:
            pred_c = (y_prob[:, c] >= t).astype(int)
            f1_c = f1_score(y_true[:, c], pred_c, zero_division=0)
            if f1_c > best_f1:
                best_f1, best_t = f1_c, t
        thresholds[c] = best_t
    return thresholds

best_thresholds = optimize_thresholds(val_labels, val_probs, NUM_CLASSES)
print("Per-class thresholds:", dict(zip(CLASSES, best_thresholds)))

2026-09-16 01:17:53.680265: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-16 01:17:53.961265: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-16 01:17:54.841482: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-16 01:17:55.031617: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-16 01:17:56.647465: E external/local_xla/xla/stream_

232/233 ━━━━━━━━━━━━━━━━━━━━ 0s 342ms/step

2026-09-16 01:19:48.128787: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-16 01:19:48.405607: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-16 01:19:49.353744: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-16 01:19:49.541844: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-16 01:19:51.305677: E external/local_xla/xla/stream_

233/233 ━━━━━━━━━━━━━━━━━━━━ 151s 490ms/step
Per-class thresholds: {'complex': np.float64(0.38), 'frog_eye_leaf_spot': np.float64(0.4), 'healthy': np.float64(0.66), 'powdery_mildew': np.float64(0.5200000000000001), 'rust': np.float64(0.4800000000000001), 'scab': np.float64(0.44000000000000006)}


In [54]:
import os

for root, dirs, files in os.walk("/kaggle/input/competitions"):
    if "train_images" in dirs:
        TRAIN_IMG_DIR = os.path.join(root, "train_images")
        print("TRAIN_IMG_DIR =", TRAIN_IMG_DIR)
        break

TRAIN_IMG_DIR = /kaggle/input/competitions/plant-pathology-2021-fgvc8/train_images


In [55]:
def apply_thresholds(y_prob, thresholds):
    return (y_prob >= thresholds[np.newaxis, :]).astype(int)

def report_metrics(y_true, y_prob, thresholds, loss_metrics, name):
    y_pred = apply_thresholds(y_prob, thresholds)
    loss, acc, prec, rec = loss_metrics[:4]
    micro_f1 = f1_score(y_true, y_pred, average="micro", zero_division=0)
    macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    per_label_f1 = f1_score(y_true, y_pred, average=None, zero_division=0)
    print(f"{name} loss: {loss:.4f}")
    print(f"{name} accuracy: {acc:.4f}")
    print(f"{name} precision: {prec:.4f}")
    print(f"{name} recall: {rec:.4f}")
    print(f"{name} micro F1: {micro_f1:.4f}")
    print(f"{name} macro F1: {macro_f1:.4f}")
    for cls, f1v in zip(CLASSES, per_label_f1):
        print(f"  {name} F1[{cls}]: {f1v:.4f}")
    return y_pred

train_eval_ds = make_dataset(train_paths, train_labels, training=False)
train_probs = model.predict(train_eval_ds)
train_loss_metrics = model.evaluate(train_eval_ds, verbose=0)
val_loss_metrics = model.evaluate(val_ds, verbose=0)

_ = report_metrics(train_labels, train_probs, best_thresholds, train_loss_metrics, "Train")
_ = report_metrics(val_labels, val_probs, best_thresholds, val_loss_metrics, "Validation")

931/932 ━━━━━━━━━━━━━━━━━━━━ 0s 353ms/step

2026-09-16 01:26:40.710853: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-16 01:26:40.935772: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-16 01:26:41.652069: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-16 01:26:41.818497: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-16 01:26:43.439247: E external/local_xla/xla/stream_

932/932 ━━━━━━━━━━━━━━━━━━━━ 353s 378ms/step
Train loss: 0.0995
Train accuracy: 0.9614
Train precision: 0.9027
Train recall: 0.8815
Train micro F1: 0.8897
Train macro F1: 0.8843
  Train F1[complex]: 0.7076
  Train F1[frog_eye_leaf_spot]: 0.8667
  Train F1[healthy]: 0.9700
  Train F1[powdery_mildew]: 0.9327
  Train F1[rust]: 0.9343
  Train F1[scab]: 0.8943
Validation loss: 0.1181
Validation accuracy: 0.9537
Validation precision: 0.8820
Validation recall: 0.8586
Validation micro F1: 0.8721
Validation macro F1: 0.8655
  Validation F1[complex]: 0.6808
  Validation F1[frog_eye_leaf_spot]: 0.8538
  Validation F1[healthy]: 0.9579
  Validation F1[powdery_mildew]: 0.9194
  Validation F1[rust]: 0.9087
  Validation F1[scab]: 0.8723


In [59]:
SAMPLE_SUB = "/kaggle/input/competitions/plant-pathology-2021-fgvc8/sample_submission.csv"
TEST_IMG_DIR = "/kaggle/input/competitions/plant-pathology-2021-fgvc8/test_images"

In [60]:
sample_sub = pd.read_csv(SAMPLE_SUB)

test_paths = (TEST_IMG_DIR + "/" + sample_sub["image"]).values
test_ds = make_dataset(test_paths, labels=None, training=False)

test_probs = model.predict(test_ds)
test_preds = apply_thresholds(test_probs, best_thresholds)

def to_label_string(row):
    labels = [CLASSES[i] for i, v in enumerate(row) if v == 1]
    if not labels:
        return "healthy"
    return " ".join(labels)

sample_sub["labels"] = [to_label_string(r) for r in test_preds]

sample_sub.to_csv("submission.csv", index=False)

sample_sub.head()

2026-09-16 01:38:51.154587: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-16 01:38:51.321221: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-16 01:38:51.781006: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-16 01:38:51.923233: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-16 01:38:53.371387: E external/local_xla/xla/stream_

1/1 ━━━━━━━━━━━━━━━━━━━━ 30s 30s/step


,image,labels
0,85f8cb619c66b863.jpg,complex scab
1,ad8770db05586b59.jpg,complex frog_eye_leaf_spot scab
2,c7b03e718489f3ca.jpg,frog_eye_leaf_spot


In [61]:
model.save("best_model.keras")

In [62]:
import os

print(os.path.exists("/kaggle/working/best_model.keras"))

True
